# Analyse exploratoire des seuils C8

Internet OFF. Noyau neuf. Aucun pixel2021. Tous les drapeaux sont désactivés. Une autorisation externe par action est nécessaire.


In [ ]:
RUN_GPU_PREFLIGHT = False
RUN_INFERENCE = False
RUN_ANALYSIS = False
RUN_FIT = False
RUN_RECOVERY = False
import os,sys
if 'torch' in sys.modules: raise RuntimeError('Redémarrer le noyau avant de fixer CUDA')
os.environ['CUDA_VISIBLE_DEVICES']='0'


In [ ]:
from pathlib import Path
import base64,hashlib,io,zipfile,json
DATA=base64.b64decode('...')
assert hashlib.sha256(DATA).hexdigest()=='c0c26d84263035d1f0afbd978546ef55a18b029e3eb9815435f639a8034c50cb'
ROOT=Path('/kaggle/working/geofirex-c8-a-code')
ROOT.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(DATA)) as z:
    for n in z.namelist():
        assert (ROOT/n).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT/'src'))
from geofirex import c8_experiments as c
print(c.preflight(ROOT))


In [ ]:
EXPECTED=set(v['map_sha256'] for v in c.read(ROOT/c.old.REPORT/'partitions.json').values())
MAPS=[p for p in Path('/kaggle/input').rglob('mount-map.json') if c.sha256(p) in EXPECTED]
print(c.preflight(ROOT,MAPS))
print(json.dumps(c.read(ROOT/c.REPORT/'budget.json'),indent=2))


Après autorisation explicite, créer une COPIE du template désactivé sous /kaggle/working, renseigner opérateur, date, quota réel, durée et destinations. Ne jamais modifier le template dans le payload. Un seul run neuf ; télécharger chaque archive vérifiée.


In [ ]:
AUTH=Path('/kaggle/working/c8_a_authorization.json')
GPU_AUTH=Path('/kaggle/working/c8_a_gpu_authorization.json')
if RUN_GPU_PREFLIGHT:
    c.authorize(ROOT,GPU_AUTH,'gpu-preflight','cuda',Path('/kaggle/working'))
    result=c.preflight(ROOT,MAPS,'/kaggle/working','cuda',gpu=True)
    c.save(Path('/kaggle/working/c8_a_gpu_preflight.json'),result)
    print(result)
else: print('Précontrôle GPU désactivé')


In [ ]:
ARCHIVES=[p for p in Path('/kaggle/input').rglob('B2_C8_V2_seed42-COMPLETE-48e35c59.*') if p.suffix in ('.zip','.bin')]
OUT=Path('/kaggle/working/'+c.A_ID)
if RUN_INFERENCE:
    assert RUN_GPU_PREFLIGHT and not RUN_FIT and not RUN_RECOVERY
    assert len(ARCHIVES)==1, 'Attacher le ZIP COMPLETE officiel intact (extension .zip ou .bin) en entrée privée'
    c.infer_thresholds(ROOT,MAPS,ARCHIVES[0],OUT,AUTH,'cuda')
else: print('Inférence exhaustive désactivée')


In [ ]:
ANALYSIS_AUTH=Path('/kaggle/working/c8_a_analysis_authorization.json')
if RUN_ANALYSIS:
    assert not RUN_FIT and not RUN_RECOVERY
    c.analyze_thresholds(ROOT,OUT/'predictions',Path('/kaggle/working/'+c.A_ID+'_ANALYSIS'),ANALYSIS_AUTH)
else: print('Analyse des seuils désactivée')


In [ ]:
from IPython.display import display,FileLink
for p in sorted(Path('/kaggle/working').glob('*.zip')):
    if p.name.startswith((c.A_ID,c.B_ID)):
        print(c.old.verify_zip(p));display(FileLink(str(p)));display(FileLink(str(p.with_suffix('.sha256'))))
